In [1]:
import json
import time

import joblib
import numpy as np
import pandas as pd
import torch

from benchmark.config import DATA_PROCESSED, DATASETS, MODELS_DIR, RESULTS_BENCHMARK
from benchmark.metrics import evaluate_regression
from benchmark.models.xrfm_model import make_processor
from benchmark.models.xrfm_model import fit_best_xrfm, fit_xrfm_cross_validation

data_name = "abalone"
cfg = DATASETS[data_name]
processed = DATA_PROCESSED / data_name
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/shado/college_project/comp9417/9417GroupProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv(processed / "train.csv")
train_df.head()

,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
0,F,0.445,0.325,0.125,0.4550,0.1785,0.1125,0.140,9
1,I,0.515,0.400,0.135,0.6965,0.3200,0.1255,0.175,9
2,I,0.565,0.440,0.175,0.8735,0.4140,0.2100,0.210,11
3,M,0.415,0.325,0.140,0.4170,0.1535,0.1015,0.144,10
4,I,0.620,0.490,0.160,1.0660,0.4460,0.2460,0.305,11


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3341 entries, 0 to 3340
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sex             3341 non-null   object 
 1   length          3341 non-null   float64
 2   diameter        3341 non-null   float64
 3   height          3341 non-null   float64
 4   whole_weight    3341 non-null   float64
 5   shucked_weight  3341 non-null   float64
 6   viscera_weight  3341 non-null   float64
 7   shell_weight    3341 non-null   float64
 8   rings           3341 non-null   int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 235.0+ KB


In [4]:
test_df = pd.read_csv(processed / "test.csv")
test_df.head()

,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
0,I,0.550,0.425,0.15,0.7665,0.3390,0.1760,0.2100,8
1,M,0.545,0.450,0.16,0.8615,0.2925,0.1545,0.3650,16
2,M,0.330,0.250,0.09,0.1970,0.0850,0.0410,0.0605,10
3,I,0.485,0.380,0.14,0.6730,0.2175,0.1300,0.1950,18
4,I,0.525,0.400,0.13,0.6445,0.3450,0.1285,0.2000,8


In [5]:
with open(processed / "metadata.json") as f:
    metadata = json.load(f)

print(metadata)

{'num_cols': ['length', 'diameter', 'height', 'whole_weight', 'shucked_weight', 'viscera_weight', 'shell_weight'], 'cat_cols': ['sex'], 'target_col': 'rings'}


In [6]:
X = train_df.drop(columns=metadata["target_col"])
y = train_df[metadata["target_col"]].to_numpy().astype(np.float32)

processor = make_processor(metadata)
n_num_cols = len(metadata["num_cols"])

In [7]:
best_trial = fit_xrfm_cross_validation(
    X, y, processor, n_num_cols, task=cfg["task"], device=device
)

print(f"Best params: {best_trial.params}")
print(f"Best MSE: {best_trial.value}")

[I 2026-04-22 15:41:33,921] A new study created in memory with name: no-name-899811ad-6c62-4561-80cc-40dfca4b6995


None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 2672, d: 10, and nval: 669
Using cheap batch size
Optimal M batch size: 2666


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]


...

None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 2673, d: 10, and nval: 668
Using expensive batch size
Optimal M batch size: 2673
Using SVD
Time taken for round 0: 0.08390140533447266 seconds
Using expensive batch size
Optimal M batch size: 2673
Using SVD
Time taken for round 1: 0.08079385757446289 seconds
Using expensive batch size
Optimal M batch size: 2673
Using SVD
Time taken for round 2: 0.08097004890441895 seconds
Using expensive batch size
Optimal M batch size: 2673
Using SVD
Time taken for round 3: 0.08126378059387207 seconds
Using expensive batch size
Optimal M batch size: 2673


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]
[I 2026-04-22 15:45:18,314] Trial 99 finished with value: 4.312680196762085 and parameters: {'exponent': 1.2499295655605867, 'kernel_type': 'kpq', 'bandwidth': 7.599681238472753, 'diag': False, 'norm_p': 1.3550153005300731, 'reg': 0.7566696295660661}. Best is trial 86 with value: 4.233418607711792.


Using SVD
Time taken for round 4: 0.08146905899047852 seconds
Tree has no split, stopping training
Using hard routing for tree prediction
Best params: {'exponent': 1.3024852816384869, 'kernel_type': 'kpq', 'bandwidth': 10.482410034390034, 'diag': False, 'norm_p': 1.454298682825814, 'reg': 0.0665834603359922}
Best MSE: 4.233418607711792


In [8]:
model, pipeline, fit_time, train_metrics, val_metrics = fit_best_xrfm(
    X,
    y,
    best_trial,
    processor,
    n_num_cols,
    task=cfg["task"],
    device=device,
)

print(f"Training time: {fit_time:.2f}s")
print(f"Train RMSE: {train_metrics['rmse']:.4f}")
print(f"Val RMSE:   {val_metrics['rmse']:.4f}")
joblib.dump(model, MODELS_DIR / data_name / "xrfm.joblib")
joblib.dump(pipeline, MODELS_DIR / data_name / "xrfm_pipeline.joblib")

None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 2672, d: 10, and nval: 669
Using expensive batch size
Optimal M batch size: 2672
Using SVD
Time taken for round 0: 0.08524918556213379 seconds
Using expensive batch size
Optimal M batch size: 2672
Using SVD
Time taken for round 1: 0.08352136611938477 seconds
Using expensive batch size
Optimal M batch size: 2672
Using SVD
Time taken for round 2: 0.08366775512695312 seconds
Using expensive batch size
Optimal M batch size: 2672
Using SVD
Time taken for round 3: 0.08358454704284668 seconds
Using expensive batch size
Optimal M batch size: 2672


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Using SVD
Time taken for round 4: 0.08348751068115234 seconds
Tree has no split, stopping training
Using hard routing for tree prediction
Using hard routing for tree prediction
Training time: 0.46s
Train RMSE: 1.9135
Val RMSE:   2.0783


['/home/shado/college_project/comp9417/9417GroupProject/models/abalone/xrfm_pipeline.joblib']

In [9]:
model = joblib.load(MODELS_DIR / data_name / "xrfm.joblib")
pipeline = joblib.load(MODELS_DIR / data_name / "xrfm_pipeline.joblib")

In [10]:
X_test = test_df.drop(columns=metadata["target_col"])
y_test = test_df[metadata["target_col"]].to_numpy().astype(np.float32)

X_test = pipeline.transform(X_test)
t0 = time.perf_counter()
y_pred = model.predict(X_test)
infer_time = (time.perf_counter() - t0) / len(X_test)

Using hard routing for tree prediction


In [11]:
results = evaluate_regression(y_test, y_pred)
print(f"RMSE: {results['rmse']:.4f}")
print(f"Inference time per sample: {infer_time * 1e6:.2f}µs")

RMSE: 2.1013
Inference time per sample: 10.11µs


In [12]:
output = {
    **results,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics,
    "fit_time_s": fit_time,
    "infer_time_per_sample_s": infer_time,
    "best_params": {
        "tuning_metric": model.tuning_metric,
        "split_method": model.split_method,
        "max_leaf_size": model.max_leaf_size,
        **model.rfm_params,
    },
}

results_dir = RESULTS_BENCHMARK / "abalone"
with open(results_dir / "xrfm.json", "w") as f:
    json.dump(output, f, indent=4)

print(f"Results saved to {results_dir / 'xrfm.json'}")

Results saved to /home/shado/college_project/comp9417/9417GroupProject/results/abalone/xrfm.json
